# Judge VQA 
### imports

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from utils.load_env_vars import load_env

load_env()

from torch.utils.data import DataLoader
import json 
import re

from tqdm import tqdm

from judge.query_sglang import query_judge_batched 
from judge.metrics import compute_count_metrics, compute_bool_metrics
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# visualize table existential and universal descriptive questions

### data loading


In [ ]:
from vqa_results_representative_utils import load_all_metrics

results = load_all_metrics()

if results:
    print(f"\nSUMMARY:")
    for dataset, df in results.items():
        print(f"{dataset.upper()}: {len(df)} metric records loaded")
else:
    print("No results loaded!")


### Helper Functions

In [ ]:
from vqa_results_representative_utils import (
    create_pivot_table,
    print_summary_stats,
    generate_latex_table,
    show_template_results,
)


### Results per Template - Clockwise

In [ ]:
show_template_results(results, "clockwise", grad_str ="gradientc")

### Results per Template - Orbit

In [ ]:
show_template_results(results, "orbit", grad_str ="gradientd")

### Results per Template - Transition

In [ ]:
show_template_results(results, "transition", grad_str="gradientd")

### Effect of added clutter on unicycle


In [ ]:
print("\nCLUTTER EFFECT - Accuracy Drop Analysis")
print("="*60)

# Create pivot tables for unicycle and unicycle_cluttered
uni = create_pivot_table(results['unicycle'])
clut = create_pivot_table(results['unicycle_cluttered'])
diff = clut - uni

print("\nAccuracy Difference (Cluttered - Unicycle):")
print(diff.to_string())
print_summary_stats(diff, "CLUTTER IMPACT SUMMARY")

# Generate LaTeX
latex = generate_latex_table(diff,
                             "Accuracy Drop: Cluttered vs Clean (Percentage Points)",
                             "tab:clutter_effect",
                             value_formatter=lambda x: f"\\gradientb{{{x:+.1f}}}")
print(f"\n{latex}")

### Tier Progression — Unicycle / Bicycle / Tricycle

In [ ]:
import matplotlib.pyplot as plt
from vqa_results_descriptive_utils import MODEL_SHORT_NAMES, model_color, build_wide

tiers = ['unicycle', 'bicycle', 'tricycle']
tier_labels = ['Unicycle', 'Bicycle', 'Tricycle']

df_wide = build_wide(results)

fig, ax = plt.subplots(figsize=(8, 5))

for model_label, row in df_wide.iterrows():
    ax.plot(range(len(tiers)), row[tiers].values, marker='o', linewidth=2,
            color=model_color(model_label), label=model_label)

ax.set_xticks(range(len(tiers)))
ax.set_xticklabels(tier_labels, fontsize=11)
ax.set_xlabel('Tier', fontsize=12)
ax.set_ylabel('Mean Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy vs. Tier (Representative questions)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from vqa_results_descriptive_utils import model_color, build_wide

tiers = ['unicycle', 'bicycle', 'tricycle']
tier_labels = ['Unicycle', 'Bicycle', 'Tricycle']

panels = [
    ('clockwise',  'Clockwise'),
    ('orbit',      'Orbit'),
    ('transition', 'Transition'),
    (None,         'All (averaged)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
fig.suptitle('Model Accuracy vs. Tier — Scatter + Correlation Line\n(Representative questions)',
             fontsize=15, y=1.01)

handles, labels = None, None

for ax, (tmpl, title) in zip(axes, panels):
    df_wide = build_wide(results, template_filter=tmpl)

    records = [
        {'model': m, 'tier_idx': ti, 'accuracy': df_wide.loc[m, tier]}
        for m, row in df_wide.iterrows()
        for ti, tier in enumerate(tiers)
    ]
    df_scatter = pd.DataFrame(records)

    for model_label, grp in df_scatter.groupby('model'):
        ax.scatter(grp['tier_idx'], grp['accuracy'], color=model_color(model_label),
                   s=80, zorder=3, label=model_label)

    x_all = df_scatter['tier_idx'].values
    y_all = df_scatter['accuracy'].values
    slope, intercept, r, p, _ = stats.linregress(x_all, y_all)
    xfit = np.linspace(-0.2, len(tiers) - 0.8, 100)
    p_label = '<0.001' if p < 0.001 else f'{p:.3f}'
    ax.plot(xfit, slope * xfit + intercept, 'k--', linewidth=2, alpha=0.75,
            label=f'Trend  r={r:.2f}, p={p_label}')

    ax.set_xticks(range(len(tiers)))
    ax.set_xticklabels(tier_labels, fontsize=11)
    ax.set_xlabel('Tier', fontsize=11)
    ax.set_ylabel('Mean Accuracy (%)', fontsize=11)
    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if handles is None:
        handles, labels = ax.get_legend_handles_labels()

fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.08), frameon=True)
plt.tight_layout()
plt.show()


### Robustness Correlation — Clutter & Nightrider

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from vqa_results_descriptive_utils import model_color, build_wide

panels = [
    ('clockwise',  'Clockwise'),
    ('orbit',      'Orbit'),
    ('transition', 'Transition'),
    (None,         'All (averaged)'),
]

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle('Model Accuracy Across Datasets with Correlation Line\n(Representative questions)',
             fontsize=15, y=1.005)

x_tiers_base = ['unicycle', 'bicycle', 'tricycle']
handles, labels = None, None

for col_idx, (tmpl, title) in enumerate(panels):

    # ── Row 0: Unicycle → Unicycle Cluttered ─────────────────────────────────
    ax = axes[0, col_idx]
    tiers = ['unicycle', 'unicycle_cluttered']
    tier_labels = ['Unicycle', 'Unicycle Cluttered']

    df_wide = build_wide(results, template_filter=tmpl, tiers=tiers)
    for model_label, row in df_wide.iterrows():
        ax.plot(range(len(tiers)), row[tiers].values, marker='o', linewidth=2,
                color=model_color(model_label), label=model_label)

    if len(df_wide) > 1:
        records = [(ti, df_wide.loc[m, t]) for m in df_wide.index for ti, t in enumerate(tiers)]
        x_all, y_all = zip(*records)
        slope, intercept, r, p, _ = stats.linregress(x_all, y_all)
        xfit = np.linspace(-0.2, len(tiers) - 0.8, 100)
        p_label = '<0.001' if p < 0.001 else f'{p:.3f}'
        ax.plot(xfit, slope * xfit + intercept, 'k--', linewidth=2, alpha=0.7,
                label=f'Trend  r={r:.2f}, p={p_label}')

    ax.set_xticks(range(len(tiers)))
    ax.set_xticklabels(tier_labels, fontsize=10)
    ax.set_xlabel('Dataset', fontsize=11)
    ax.set_ylabel('Mean Accuracy (%)', fontsize=11)
    ax.set_title(f'{title}\nUnicycle → Cluttered', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if handles is None:
        handles, labels = ax.get_legend_handles_labels()

    # ── Row 1: Avg(Uni/Bi/Tri) → Nightrider ──────────────────────────────────
    ax = axes[1, col_idx]
    df_base  = build_wide(results, template_filter=tmpl, tiers=x_tiers_base)
    df_night = build_wide(results, template_filter=tmpl, tiers=['nightrider'])

    common = df_base.index.intersection(df_night.index)
    df_base  = df_base.loc[common]
    df_night = df_night.loc[common]

    x_avg   = df_base.mean(axis=1)
    y_night = df_night['nightrider']
    df_p2   = pd.DataFrame({'Avg(Uni/Bi/Tri)': x_avg, 'Nightrider': y_night})

    for model_label, row in df_p2.iterrows():
        ax.plot([0, 1], row.values, marker='o', linewidth=2,
                color=model_color(model_label), label=model_label)

    if len(common) > 1:
        records2 = [(0, x_avg[m]) for m in common] + [(1, y_night[m]) for m in common]
        x2, y2 = zip(*records2)
        slope2, intercept2, r2, p2, _ = stats.linregress(x2, y2)
        xfit2 = np.linspace(-0.2, 1.2, 100)
        p_label2 = '<0.001' if p2 < 0.001 else f'{p2:.3f}'
        ax.plot(xfit2, slope2 * xfit2 + intercept2, 'k--', linewidth=2, alpha=0.7,
                label=f'Trend  r={r2:.2f}, p={p_label2}')

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Avg(Uni/Bi/Tri)', 'Nightrider'], fontsize=10)
    ax.set_xlabel('Dataset', fontsize=11)
    ax.set_ylabel('Mean Accuracy (%)', fontsize=11)
    ax.set_title(f'{title}\nAvg(Uni/Bi/Tri) → Nightrider', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.06), frameon=True)
plt.tight_layout()
plt.show()


### Compact Overview — Tier Scatter & Nightrider (2×3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from matplotlib.lines import Line2D
from vqa_results_descriptive_utils import model_color, model_marker, build_wide

tiers = ['unicycle', 'bicycle', 'tricycle']
tier_labels = ['Unicycle', 'Bicycle', 'Tricycle']

templates = [
    ('clockwise',  'Orbit Direction'),
    ('orbit',      'Orbit Center'),
    ('transition', 'Cyclic Transition'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharey=False)


handles, labels = None, None

for col_idx, (tmpl, title) in enumerate(templates):

    # ── Row 0: Uni / Bi / Tricycle scatter + OLS ─────────────────────────────
    ax = axes[0, col_idx]
    df_wide = build_wide(results, template_filter=tmpl)

    records = [
        {'model': m, 'tier_idx': ti, 'accuracy': df_wide.loc[m, tier]}
        for m, row in df_wide.iterrows()
        for ti, tier in enumerate(tiers)
    ]
    df_scatter = pd.DataFrame(records)

    for model_label, grp in df_scatter.groupby('model'):
        ax.plot(grp['tier_idx'], grp['accuracy'], marker='o', linewidth=1.5,
                color=model_color(model_label), alpha=0.6, zorder=2)
        ax.scatter(grp['tier_idx'], grp['accuracy'], color=model_color(model_label),
                   marker=model_marker(model_label), s=80, zorder=3, label=model_label)

    x_all = df_scatter['tier_idx'].values
    y_all = df_scatter['accuracy'].values
    slope, intercept, r, p, _ = stats.linregress(x_all, y_all)
    xfit = np.linspace(-0.2, len(tiers) - 0.8, 100)
    p_label = '<0.001' if p < 0.001 else f'{p:.3f}'
    ax.plot(xfit, slope * xfit + intercept, 'k--', linewidth=2, alpha=0.75,
            label='_nolegend_')
    ax.spines[['right','top']].set_visible(False)

    ax.set_xticks(range(len(tiers)))
    ax.set_xticklabels(tier_labels, fontsize=19)
    ax.tick_params(axis='y', labelsize=19)
    if col_idx == 0:
        ax.set_ylabel('Accuracy (%)', fontsize=20)
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=6))
    else:
        ax.tick_params(axis='y', labelleft=True)
    ax.set_title(f'{title}', fontsize=20, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if handles is None:
        handles, labels = ax.get_legend_handles_labels()

    # ── Row 1: Avg(Uni/Bi/Tri) → Nightrider ──────────────────────────────────
    ax = axes[1, col_idx]
    
    df_base  = build_wide(results, template_filter=tmpl, tiers=['unicycle', 'bicycle', 'tricycle'])
    df_night = build_wide(results, template_filter=tmpl, tiers=['nightrider'])

    common = df_base.index.intersection(df_night.index)
    _weights = pd.Series({'unicycle': 160, 'bicycle': 240, 'tricycle': 280})
    _weights = _weights / _weights.sum()
    x_avg = df_base.loc[common].mul(_weights).sum(axis=1)
    y_night = df_night.loc[common, 'nightrider']
    df_p2   = pd.DataFrame({'Avg(Uni/Bi/Tri)': x_avg, 'Nightrider': y_night})

    for model_label, row in df_p2.iterrows():
        ax.plot([0, 1], row.values, linewidth=1.5,
                color=model_color(model_label), alpha=0.6, zorder=2)
        ax.scatter([0, 1], row.values, color=model_color(model_label),
                   marker=model_marker(model_label), s=80, zorder=3, label=model_label)

    if len(common) > 1:
        records2 = [(0, x_avg[m]) for m in common] + [(1, y_night[m]) for m in common]
        x2, y2 = zip(*records2)
        slope2, intercept2, r2, p2, _ = stats.linregress(x2, y2)
        xfit2 = np.linspace(-0.2, 1.2, 100)
        p_label2 = '<0.001' if p2 < 0.001 else f'{p2:.3f}'
        ax.plot(xfit2, slope2 * xfit2 + intercept2, 'k--', linewidth=2, alpha=0.7,
                label='_nolegend_')
    ax.spines[['right','top']].set_visible(False)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Avg(Uni/Bi/Tri)', 'Nightrider'], fontsize=19)
    ax.tick_params(axis='y', labelsize=19)
    if col_idx == 0:
        ax.set_ylabel('Accuracy (%)', fontsize=20)
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=6))
    else:
        ax.tick_params(axis='y', labelleft=True)    # ax.set_title(f'{title}', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)

# Reorder: LLaVA → Gemini 2.5 → Gemini 2.0 → Intern → OLS
_llava  = [(h, l) for h, l in zip(handles, labels) if 'LLaVA' in l]
_g25    = [(h, l) for h, l in zip(handles, labels) if 'Gemini' in l and '2.5' in l]
_g20    = [(h, l) for h, l in zip(handles, labels) if 'Gemini' in l and '2.0' in l]
_intern = [(h, l) for h, l in zip(handles, labels) if 'Intern' in l]
handles, labels = zip(*(_llava + _g25 + _g20 + _intern))
handles = list(handles) + [Line2D([0], [0], color='k', linewidth=2, linestyle='--', alpha=0.75)]
labels  = list(labels)  + ['OLS trend']

fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=18,
           bbox_to_anchor=(0.5, -0.15), frameon=True)
plt.tight_layout()
plt.savefig("ablation.pdf", bbox_inches="tight")
plt.show()
